In [ ]:
import os
import json
import sqlite3
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OPENAI API Key not set")

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()    
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def set_ticket_price(destination_city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', 
            (destination_city.lower(), price, price)
        )
        conn.commit()

def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"The price of a ticket to {destination_city} is ${result[0]}" if result else f"No pricing metadata found for {destination_city}."

initial_fares = {"london": 799, "paris": 899, "tokyo": 1420, "berlin": 499}
for city, price in initial_fares.items():
    set_ticket_price(city, price)

In [ ]:
price_function_schema = {
    "name": "get_ticket_price",
    "description": "Retrieve the current live flight ticket price or fare for a specified destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The target city name, e.g., London, Tokyo, Paris.",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function_schema = {
    "name": "set_ticket_price",
    "description": "Set, update, or overwrite the numeric flight ticket price or fare record for a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The exact city target destination to update in the database."
            },
            "price": {
                "type": "number",
                "description": "The new numeric fare amount value (e.g., 1350 or 999.5)."
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

price_tool = {"type": "function", "function": price_function_schema}
set_price_tool = {"type": "function", "function": set_price_function_schema}
tools = [price_tool, set_price_tool]

TOOL_DICTIONARY = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price
}

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        tool_function = TOOL_DICTIONARY.get(tool_name)
        
        if tool_function:
            arguments = json.loads(tool_call.function.arguments)
            try:
                execution_result = tool_function(**arguments)
                if execution_result is None:
                    execution_result = f"Successfully executed state change mutation tool: {tool_name}."
            except TypeError as e:
                execution_result = f"Error: Tool execution signature collision: {str(e)}"
            
            responses.append({
                "role": "tool",
                "content": str(execution_result),
                "tool_call_id": tool_call.id
            })
        else:
            responses.append({
                "role": "tool",
                "content": f"Error: Unregistered tool signature request received for '{tool_name}'.",
                "tool_call_id": tool_call.id
            })
    return responses

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers.
Always be accurate. If you don't know the answer, say so.
"""

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    max_iterations = 5
    iteration = 0

    while response.choices[0].finish_reason == "tool_calls" and iteration < max_iterations:
        assistant_msg = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_msg)
        
        messages.append(assistant_msg)
        messages.extend(tool_responses)
        
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        iteration += 1
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()